In [1]:
import sys 
import os 

sys.path.append(os.path.join('..', '..'))

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

## 1. Carregamento das bases

Serão utilizadas três bases limpas:

- `data/sigesguarda/cleaned/base_unificada.csv`: ocorrências registradas pela SIGESGUARDA;
- `data/ibge2010/cleaned/base_bairros_2010.csv`: indicadores socioeconômicos por bairro do censo 2010;
- `data/ibge2022/cleaned/base_bairros_2022.csv`: indicadores socioeconômicos por bairro do censo 2022.

In [3]:
ANO_INICIAL = 2009
ANO_FINAL = 2026
ANO_CENSO_2010 = 2010
ANO_CENSO_2022 = 2022

In [4]:
data_dir = Path("../../data/")

arquivo_sigesguarda = data_dir / "sigesguarda/cleaned/base_unificada.csv"
arquivo_ibge2010 = data_dir / "ibge2010/cleaned/base_bairros_2010.csv"
arquivo_ibge2022 = data_dir / "ibge2022/cleaned/base_bairros_2022.csv"

In [5]:
sigesguarda = pd.read_csv(arquivo_sigesguarda, sep=";", dtype=str)
base_2010 = pd.read_csv(arquivo_ibge2010, sep=",", dtype=str)
base_2022 = pd.read_csv(arquivo_ibge2022, sep=",", dtype=str)

Como os nomes dos bairros são a chave principal para garantir a unificação das bases de dados, vamos garantir que todos os nomes estão padronizados da mesma forma:

In [6]:
bairros_2010 = set(base_2010["ATENDIMENTO_BAIRRO_NOME"].dropna().astype(str).str.strip().str.lower())
bairros_2022 = set(base_2022["ATENDIMENTO_BAIRRO_NOME"].dropna().astype(str).str.strip().str.lower())

In [7]:
print(bairros_2022 == bairros_2010)

True


## 2. Mergeando dados do censo de 2010 e de 2022

In [8]:
censo = base_2010.merge(
    base_2022,
    on="ATENDIMENTO_BAIRRO_NOME",
    how="inner"
)

In [9]:
censo = censo.rename(columns={
    "domicilios_particulares_permanentes_2010": "domicilios_particulares_ocupados_2010",
    "domicilios_particulares_permanentemente_ocupados_2022": "domicilios_particulares_ocupados_2022",
})

In [10]:
censo.head(3)

,ATENDIMENTO_BAIRRO_NOME,populacao_2010,pessoas_10_anos_ou_mais_2010,pct_sem_rendimento_2010,pct_rendimento_ate_1_sm_2010,pct_rendimento_ate_2_sm_2010,pct_rendimento_acima_5_sm_2010,resp_domicilios_particulares_2010,rendimento_medio_responsavel_2010,rendimento_medio_responsavel_sm_2010,...,sem_banheiro_sanitario_2022,pct_sem_banheiro_sanitario_2022,esgotamento_precario_2022,pct_esgotamento_precario_2022,sem_rede_geral_agua_2022,pct_sem_rede_geral_agua_2022,lixo_destino_inadequado_2022,pct_lixo_destino_inadequado_2022,domicilios_improvisados_estrutura_degradada_2022,pct_domicilios_improvisados_estrutura_degradada_2022
0,abranches,13151,11401.0,30.988509779843877,13.147969476361723,38.636961670028946,10.17454609244803,4067.0,2066.70258261594,4.052358005129294,...,0,0.0,36,0.7092198581560284,13,0.256107171000788,7,0.13790386130811663,0,0.0
1,agua verde,51290,47282.0,25.472695740450913,4.445666426970094,15.75440971194112,36.21462713083203,19677.0,5437.484683309714,10.661734673156301,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,ahu,11479,10456.0,27.926549349655698,4.982785003825555,15.885615914307575,35.52027543993879,4214.0,5550.206715634838,10.882758265950661,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


Algumas variáveis do censo de 2010 não possuem correspondente no censo de 2022 e serão dropadas:

| Coluna em 2010 | Motivo |
|----------------|--------|
| `pessoas_10_anos_ou_mais_2010` | não há variável equivalente de 10 anos ou mais em 2022 |
| `pct_sem_rendimento_2010` | não há faixa equivalente em 2022 |
| `pct_rendimento_ate_1_sm_2010` | não há faixa equivalente em 2022 |
| `pct_rendimento_ate_2_sm_2010` | não há faixa equivalente em 2022 |
| `pct_rendimento_acima_5_sm_2010` | não há faixa equivalente em 2022 |
| `pop_10mais_2010` | em 2022 temos apenas `pop_15mais_2022` |
| `alfabetizados_10mais_2010` |	em 2022 temos apenas `alfabetizados_15mais_2022` |
| `pct_alfabetizacao_10mais_2010` |	em 2022 temos apenas taxa para 15 anos ou mais |
| `pct_analfabetismo_10mais_2010` | em 2022 temos apenas taxa para 15 anos ou mais |


Para o censo de 2022:

| Coluna em 2022 | Motivo | 
| -------------- | ------ |
| `domicilios_improvisados_estrutura_degradada_2022` | não há variável equivalente extraída em 2010 |
| `pct_domicilios_improvisados_estrutura_degradada_2022`| não há variável equivalente extraída em 2010 |

In [11]:
colunas_sem_correspondente_2022 = [
    "pessoas_10_anos_ou_mais_2010",
    "pct_sem_rendimento_2010",
    "pct_rendimento_ate_1_sm_2010",
    "pct_rendimento_ate_2_sm_2010",
    "pct_rendimento_acima_5_sm_2010",
    "pop_10mais_2010",
    "alfabetizados_10mais_2010",
    "pct_alfabetizacao_10mais_2010",
    "pct_analfabetismo_10mais_2010",
    "rendimento_medio_responsavel_2010"
]

In [12]:
colunas_sem_correspondente_2010 = [
    "moradores_domicilios_particulares_2022",
    "rendimento_medio_responsavel_2022",
    "domicilios_improvisados_estrutura_degradada_2022",
    "pct_domicilios_improvisados_estrutura_degradada_2022"
]

In [13]:
censo = censo.drop(
    columns=colunas_sem_correspondente_2022,
    errors="ignore"
)

In [14]:
censo = censo.drop(
    columns=colunas_sem_correspondente_2010,
    errors="ignore"
)

In [15]:
censo.head(3)

,ATENDIMENTO_BAIRRO_NOME,populacao_2010,resp_domicilios_particulares_2010,rendimento_medio_responsavel_sm_2010,pop_15mais_2010,alfabetizados_15mais_2010,pct_alfabetizacao_15mais_2010,pct_analfabetismo_15mais_2010,domicilios_particulares_ocupados_2010,sem_rede_geral_agua_2010,...,analfabetos_15mais_2022,domicilios_particulares_ocupados_2022,sem_banheiro_sanitario_2022,pct_sem_banheiro_sanitario_2022,esgotamento_precario_2022,pct_esgotamento_precario_2022,sem_rede_geral_agua_2022,pct_sem_rede_geral_agua_2022,lixo_destino_inadequado_2022,pct_lixo_destino_inadequado_2022
0,abranches,13151,4067.0,4.052358005129294,10299.0,10072.0,97.79590251480727,2.204097485192733,4064,79.0,...,201,5076,0,0.0,36,0.7092198581560284,13,0.256107171000788,7,0.13790386130811663
1,agua verde,51290,19677.0,10.661734673156301,44938.0,44786.0,99.66175619742756,0.33824380257243547,19670,93.0,...,129,21971,0,0.0,0,0.0,0,0.0,0,0.0
2,ahu,11479,4214.0,10.882758265950661,9849.0,9819.0,99.69540054827901,0.3045994517209891,4213,83.0,...,20,4858,0,0.0,0,0.0,0,0.0,0,0.0


## 3. Interpolação e extrapolação linear

Os dados socioeconômicos utilizados neste trabalho são provenientes dos censos demográficos de 2010 e 2022. Já os dados do SIGESGUARDA abrangem o período de 2009 a 2026. Dessa forma, como as bases possuem coberturas temporais diferentes, é necessário estimar os valores socioeconômicos para os anos em que não há dados censitários observados.

Para os anos compreendidos entre 2010 e 2022, será utilizada a **interpolação linear**, pois esses anos estão dentro do intervalo delimitado pelos dois censos disponíveis. A interpolação consiste em estimar valores intermediários entre dois pontos conhecidos, assumindo uma variação linear entre eles.

Matematicamente, para um ano \(x_*\) entre 2010 e 2022, temos:

$$
y(x_*) = y_{2010} + \frac{x_* - 2010}{2022 - 2010}(y_{2022} - y_{2010})
$$

em que:

- $y(x_*)$ é o valor estimado para o ano desejado;
- $y_{2010}$ é o valor observado no censo de 2010;
- $y_{2022}$ é o valor observado no censo de 2022;
- $x_*$ é o ano para o qual se deseja estimar o valor.

Para os anos fora do intervalo censitário, isto é, 2009 e os anos de 2023 em diante, será utilizada a **extrapolação linear**. A extrapolação é o processo de estimar valores fora dos limites dos pontos conhecidos. 

Neste caso, a extrapolação será feita a partir da mesma tendência linear observada entre os censos de 2010 e 2022:

$$
y(x_*) = y_{2010} + \frac{x_* - 2010}{2022 - 2010}(y_{2022} - y_{2010})
$$

Quando $x_{*} < 2010$, como no caso de 2009, temos uma extrapolação para trás. Quando $x_{*} > 2022$, como em 2023, 2024, 2025 e 2026, temos uma extrapolação para frente.

Fonte: 

https://ufsj.edu.br/portal-repositorio/File/nepomuceno/mn/12MN_Interpola.pdf

https://www.zohry.com/cdc103/Lecture%2010%20-%2022%20Apr%202012.pdf

https://www.measureevaluation.org/resources/training/online-courses-and-resources/non-certificate-courses-and-mini-tutorials/population-analysis-for-planners/lesson-6.html


In [16]:
anos_estimados = np.arange(2009, 2027)
anos_censo = np.array([2010, 2022])

In [17]:
censo.columns

Index(['ATENDIMENTO_BAIRRO_NOME', 'populacao_2010',
       'resp_domicilios_particulares_2010',
       'rendimento_medio_responsavel_sm_2010', 'pop_15mais_2010',
       'alfabetizados_15mais_2010', 'pct_alfabetizacao_15mais_2010',
       'pct_analfabetismo_15mais_2010',
       'domicilios_particulares_ocupados_2010', 'sem_rede_geral_agua_2010',
       'pct_sem_rede_geral_agua_2010', 'sem_banheiro_sanitario_2010',
       'pct_sem_banheiro_sanitario_2010', 'esgotamento_precario_2010',
       'pct_esgotamento_precario_2010', 'lixo_destino_inadequado_2010',
       'pct_lixo_destino_inadequado_2010', 'populacao_2022',
       'resp_domicilios_particulares_2022',
       'rendimento_medio_responsavel_sm_2022', 'pop_15mais_2022',
       'alfabetizados_15mais_2022', 'pct_alfabetizacao_15mais_2022',
       'pct_analfabetismo_15mais_2022', 'analfabetos_15mais_2022',
       'domicilios_particulares_ocupados_2022', 'sem_banheiro_sanitario_2022',
       'pct_sem_banheiro_sanitario_2022', 'esgotamento

Variáveis que foram obtidas diretamente, como:

- `populacao`;
- `resp_domiclios_particulares`;
- `rendimento_medio_responsavel_sm`;
- `rendimento_mediO_responsavel_sm_2022`;
- `pop_15mais`;
- `alfabetizados_15_mais`;
- `domicilios_particulares_ocupados`;
- `sem_banheiro_sanitario`;
- `esgotamento_precario`;
- `sem_rede_geral_agua`;
- `lixo_destino_inadequado`

serão obtidas por meio da interpolação ou extrapolação. Já variáveis que são calculadas as variáveis já descritas, serão recalculadas, como:

- `analfabetos_15mais`;
- `pct_alfabetizacao_15mais`;
- `pct_analfabetismo_15mais`;
- `pct_sem_banheiro_sanitario`;
- `pct_esgotamento_precario`;
- `pct_sem_rede_geral_agua`;
- `pct_lixo_destino_inadequado`.

Obs: 

Ao extrapolar separadamente `pop_15mais` e `alfabetizados_15mais`, alguns anos posteriores a 2022 geraram inconsistências lógicas. Nos testes realizados, o bairro Juvevê apresentou esse problema em 2025 e 2026, a estimativa de `alfabetizados_15mais` ficou maior que a estimativa `pop_15mais`, produzindo `analfabetos_15mais` negativo e `pct_alfabetizacao_15mais` acima de 100%. 

Para evitar esse tipo de resultado, a taxa de alfabetizacao dos anos posteriores a 2022 será fixada no valor observado no censo de 2022. A partir dessa taxa fixada, recalculamos `alfabetizados_15mais`, `analfabetos_15mais` e `pct_analfabetismo_15mais`.

In [18]:
variaveis_interpolar = [
    "populacao",
    "resp_domicilios_particulares",
    "rendimento_medio_responsavel_sm",
    "pop_15mais",
    "alfabetizados_15mais",
    "domicilios_particulares_ocupados",
    "sem_banheiro_sanitario",
    "esgotamento_precario",
    "sem_rede_geral_agua",
    "lixo_destino_inadequado"
]

In [19]:
for var in variaveis_interpolar:
    for ano in [ANO_CENSO_2010, ANO_CENSO_2022]:
        coluna = f"{var}_{ano}"
        censo[coluna] = pd.to_numeric(censo[coluna], errors="coerce")

In [20]:
def estimar_linear(valor_2010, valor_2022, anos_estimado):
    if pd.isna(valor_2010) or pd.isna(valor_2022):
        return np.repeat(np.nan, len(anos_estimados))
    
    x = np.array([ANO_CENSO_2010, ANO_CENSO_2022])
    y = np.array([valor_2010, valor_2022], dtype=float)

    f = interp1d(
        x, 
        y,
        kind="linear",
        fill_value="extrapolate"
    )

    return f(anos_estimados)

In [21]:
anos_estimados = np.arange(ANO_INICIAL, ANO_FINAL + 1)

In [22]:
linhas = []

for _, row in censo.iterrows():
    bairro = row["ATENDIMENTO_BAIRRO_NOME"]

    estimativas = {}

    for var in variaveis_interpolar:
        estimativas[var] = estimar_linear(
            row[f"{var}_{ANO_CENSO_2010}"],
            row[f"{var}_{ANO_CENSO_2022}"],
            anos_estimados
        )

    for i, ano in enumerate(anos_estimados):
        nova_linha = {
            "ATENDIMENTO_BAIRRO_NOME": bairro,
            "ano": int(ano),
        }

        if ano in [ANO_CENSO_2010, ANO_CENSO_2022]:
            nova_linha["tipo_estimativa"] = "observado"
        elif ANO_CENSO_2010 < ano < ANO_CENSO_2022:
            nova_linha["tipo_estimativa"] = "interpolado"
        else: 
            nova_linha["tipo_estimativa"] = "extrapolado"

        for var, valores in estimativas.items():
            nova_linha[f"{var}_estimado"] = valores[i]

        linhas.append(nova_linha)

In [23]:
base_socio_anual = pd.DataFrame(linhas)

In [24]:
colunas_saneamento_absolutas = [
    "sem_banheiro_sanitario_estimado",
    "esgotamento_precario_estimado",
    "sem_rede_geral_agua_estimado",
    "lixo_destino_inadequado_estimado"
]

In [25]:
base_socio_anual["analfabetos_15mais_estimado"] = (
    base_socio_anual["pop_15mais_estimado"]
    - base_socio_anual["alfabetizados_15mais_estimado"]
)

In [26]:
base_socio_anual["pct_alfabetizacao_15mais_estimado"] = (
    base_socio_anual["alfabetizados_15mais_estimado"]
    / base_socio_anual["pop_15mais_estimado"]
    * 100
)

Fixando a taxa de alfabetizacao em 2022:

In [27]:
taxa_alfabetizacao_2022 = (
    censo[["ATENDIMENTO_BAIRRO_NOME", "pct_alfabetizacao_15mais_2022"]]
    .copy()
)

In [28]:
taxa_alfabetizacao_2022["pct_alfabetizacao_15mais_2022"] = pd.to_numeric(
    taxa_alfabetizacao_2022["pct_alfabetizacao_15mais_2022"],
    errors="coerce"
)

In [29]:
base_socio_anual = base_socio_anual.merge(
    taxa_alfabetizacao_2022,
    on="ATENDIMENTO_BAIRRO_NOME",
    how="left"
)

In [30]:
mask_pos_2022 = base_socio_anual["ano"] > ANO_CENSO_2022

In [31]:
base_socio_anual.loc[
    mask_pos_2022,
    "pct_alfabetizacao_15mais_estimado"
] = base_socio_anual.loc[
    mask_pos_2022,
    "pct_alfabetizacao_15mais_2022"
]

In [32]:
base_socio_anual["pct_alfabetizacao_15mais_estimado"] = (
    base_socio_anual["pct_alfabetizacao_15mais_estimado"]
    .clip(lower=0, upper=100)
)

In [33]:
base_socio_anual["alfabetizados_15mais_estimado"] = (
    base_socio_anual["pop_15mais_estimado"]
    * base_socio_anual["pct_alfabetizacao_15mais_estimado"]
    / 100
)

In [34]:
base_socio_anual["analfabetos_15mais_estimado"] = (
    base_socio_anual["pop_15mais_estimado"]
    - base_socio_anual["alfabetizados_15mais_estimado"]
)

In [35]:
base_socio_anual = base_socio_anual.drop(
    columns=["pct_alfabetizacao_15mais_2022"]
)

In [36]:
base_socio_anual["pct_analfabetismo_15mais_estimado"] = (
    base_socio_anual["analfabetos_15mais_estimado"]
    / base_socio_anual["pop_15mais_estimado"]
    * 100
)

In [37]:
base_socio_anual.loc[
    base_socio_anual["pop_15mais_estimado"] <= 0,
    [ 
        "pct_alfabetizacao_15mais_estimado",
        "pct_analfabetismo_15mais_estimado"
    ]
] = np.nan

In [38]:
base_socio_anual["domicilios_particulares_ocupados_estimado"] = (
    base_socio_anual["domicilios_particulares_ocupados_estimado"]
    .clip(lower=0)
)

In [39]:
base_socio_anual[colunas_saneamento_absolutas] = (
    base_socio_anual[colunas_saneamento_absolutas]
    .clip(lower=0)
)

In [40]:
for coluna in colunas_saneamento_absolutas:
    base_socio_anual[coluna] = base_socio_anual[coluna].where(
        base_socio_anual[coluna] <= base_socio_anual["domicilios_particulares_ocupados_estimado"],
        base_socio_anual["domicilios_particulares_ocupados_estimado"]
    )

In [41]:
variaveis_saneamento = [
    "sem_banheiro_sanitario",
    "esgotamento_precario",
    "sem_rede_geral_agua",
    "lixo_destino_inadequado"
]

In [42]:
for var in variaveis_saneamento:
    base_socio_anual[f"pct_{var}_estimado"] = (
        base_socio_anual[f"{var}_estimado"] /
        base_socio_anual["domicilios_particulares_ocupados_estimado"]
    ) * 100

In [43]:
base_socio_anual.loc[
    base_socio_anual["domicilios_particulares_ocupados_estimado"] <= 0,
    [f"pct_{var}_estimado" for var in variaveis_saneamento]
] = np.nan

In [44]:
assert len(base_socio_anual) == len(bairros_2010) * len(anos_estimados)

In [45]:
base_socio_anual.head(3)

,ATENDIMENTO_BAIRRO_NOME,ano,tipo_estimativa,populacao_estimado,resp_domicilios_particulares_estimado,rendimento_medio_responsavel_sm_estimado,pop_15mais_estimado,alfabetizados_15mais_estimado,domicilios_particulares_ocupados_estimado,sem_banheiro_sanitario_estimado,esgotamento_precario_estimado,sem_rede_geral_agua_estimado,lixo_destino_inadequado_estimado,analfabetos_15mais_estimado,pct_alfabetizacao_15mais_estimado,pct_analfabetismo_15mais_estimado,pct_sem_banheiro_sanitario_estimado,pct_esgotamento_precario_estimado,pct_sem_rede_geral_agua_estimado,pct_lixo_destino_inadequado_estimado
0,abranches,2009,extrapolado,13081.0,3982.916667,4.038095,10201.5,9972.333333,3979.666667,3.25,1097.666667,84.5,7.0,229.166667,97.753598,2.246402,0.081665,27.581875,2.123293,0.175894
1,abranches,2010,observado,13151.0,4067.000000,4.052358,10299.0,10072.000000,4064.000000,3.00,1016.000000,79.0,7.0,227.000000,97.795903,2.204097,0.073819,25.000000,1.943898,0.172244
2,abranches,2011,interpolado,13221.0,4151.083333,4.066621,10396.5,10171.666667,4148.333333,2.75,934.333333,73.5,7.0,224.833333,97.837413,2.162587,0.066292,22.523102,1.771796,0.168742


Agora, aproximaremos para o inteiro mais próximo os dados das colunas:

- `populacao_estimado`;
- `resp_domicilios_particulares_estimado`;
- `pop_15mais_estimado`;
- `alfabetizados_15mais_estimado`;
- `analfabetos_15mais_estimado`
- `domicilios_particulares_ocupados_estimado`;
- `sem_banheiro_sanitario_estimado`;
- `esgotamento_precario_estimado`;
- `sem_rede_geral_agua_estimado`;
- `lixo_destino_inadequado_estimado`.

Enquanto as seguintes variáveis serão aproximadas para duas casas decimais:

- `rendimento_medio_responsavel_sm_estimado`;
- `pct_alfabetizacao_15mais_estimado`;
- `pct_analfabetismo_15mais_estimado`;
- `pct_sem_banheiro_sanitario_estimado`;
- `pct_esgotamento_precario_estimado`;
- `pct_sem_rede_geral_agua_estimado`;
- `pct_lixo_destino_inadequado_estimado`.

In [46]:
colunas_inteiras = [
    "populacao_estimado",
    "resp_domicilios_particulares_estimado",
    "pop_15mais_estimado",
    "alfabetizados_15mais_estimado",
    "analfabetos_15mais_estimado",
    "domicilios_particulares_ocupados_estimado",
    "sem_banheiro_sanitario_estimado",
    "esgotamento_precario_estimado",
    "sem_rede_geral_agua_estimado",
    "lixo_destino_inadequado_estimado"
]

In [47]:
colunas_duas_casas = [
    "rendimento_medio_responsavel_sm_estimado",
    "pct_alfabetizacao_15mais_estimado",
    "pct_analfabetismo_15mais_estimado",
    "pct_sem_banheiro_sanitario_estimado",
    "pct_esgotamento_precario_estimado",
    "pct_sem_rede_geral_agua_estimado",
    "pct_lixo_destino_inadequado_estimado",
]

In [48]:
base_socio_anual[colunas_inteiras] = (
    base_socio_anual[colunas_inteiras]
    .round(0)
    .astype("Int64")
)

In [49]:
base_socio_anual[colunas_duas_casas] = (
    base_socio_anual[colunas_duas_casas]
    .round(2)
)

In [50]:
base_socio_anual.head(3)

,ATENDIMENTO_BAIRRO_NOME,ano,tipo_estimativa,populacao_estimado,resp_domicilios_particulares_estimado,rendimento_medio_responsavel_sm_estimado,pop_15mais_estimado,alfabetizados_15mais_estimado,domicilios_particulares_ocupados_estimado,sem_banheiro_sanitario_estimado,esgotamento_precario_estimado,sem_rede_geral_agua_estimado,lixo_destino_inadequado_estimado,analfabetos_15mais_estimado,pct_alfabetizacao_15mais_estimado,pct_analfabetismo_15mais_estimado,pct_sem_banheiro_sanitario_estimado,pct_esgotamento_precario_estimado,pct_sem_rede_geral_agua_estimado,pct_lixo_destino_inadequado_estimado
0,abranches,2009,extrapolado,13081,3983,4.04,10202,9972,3980,3,1098,84,7,229,97.75,2.25,0.08,27.58,2.12,0.18
1,abranches,2010,observado,13151,4067,4.05,10299,10072,4064,3,1016,79,7,227,97.80,2.20,0.07,25.00,1.94,0.17
2,abranches,2011,interpolado,13221,4151,4.07,10396,10172,4148,3,934,74,7,225,97.84,2.16,0.07,22.52,1.77,0.17


## 4. Mergeando a base de dados final

Agora que os dados foram interpolados, vamos mergear a base de dados do censo com a base de dados dos SIGESGUARDA.

In [51]:
sigesguarda["OCORRENCIA_ANO"] = sigesguarda["OCORRENCIA_ANO"].astype(int)
base_socio_anual["ano"] = base_socio_anual["ano"].astype(int)

In [52]:
colunas_socio = [
    "ATENDIMENTO_BAIRRO_NOME",
    "ano",
    "tipo_estimativa",
    "populacao_estimado",
    "resp_domicilios_particulares_estimado",
    "rendimento_medio_responsavel_sm_estimado",
    "pop_15mais_estimado",
    "alfabetizados_15mais_estimado",
    "analfabetos_15mais_estimado",
    "pct_alfabetizacao_15mais_estimado",
    "pct_analfabetismo_15mais_estimado",
    "domicilios_particulares_ocupados_estimado",
    "sem_banheiro_sanitario_estimado",
    "pct_sem_banheiro_sanitario_estimado",
    "esgotamento_precario_estimado",
    "pct_esgotamento_precario_estimado",
    "sem_rede_geral_agua_estimado",
    "pct_sem_rede_geral_agua_estimado",
    "lixo_destino_inadequado_estimado",
    "pct_lixo_destino_inadequado_estimado"
]

In [53]:
base_socio_merge = base_socio_anual[colunas_socio].copy()

In [54]:
base_final = sigesguarda.merge(
    base_socio_merge,
    left_on=["ATENDIMENTO_BAIRRO_NOME", "OCORRENCIA_ANO"],
    right_on=["ATENDIMENTO_BAIRRO_NOME", "ano"],
    how="left",
    validate="many_to_one"
)

In [55]:
print(len(base_final) == len(sigesguarda))

True


In [56]:
output_dir = data_dir / "base_de_dados_final"
arquivo_saida = output_dir / "base_sigesguarda_socioeconomica.csv"

In [57]:
base_final.to_csv(
    arquivo_saida,
    sep=";",
    index=False,
    encoding="latin1"
)

In [58]:
arquivo_saida

PosixPath('../../data/base_de_dados_final/base_sigesguarda_socioeconomica.csv')